### LIBRARY IMPORTS

In [1]:
import pandas as pd
import numpy as np
import os
import torch
import torch.nn as nn
import torch.optim as optim
import copy


from sklearn.metrics import accuracy_score

if os.getcwd().endswith('notebooks'):
    os.chdir('..')

from src import *
from src.data_manager import DataManager
from src.processor import Processor
from src.cnn_regressor import CNNRegressor

### CONFIGURATION

In [4]:
data_manager = DataManager()

datasets_config, modeling_config = data_manager.load_config()

active_dataset = modeling_config["main"]["active_dataset"]
active_dataset_config = datasets_config[active_dataset]

problem_type = active_dataset_config["problem_type"]

gradient_boosting_config = modeling_config["gradient_boosting"]

weak_learner_key = gradient_boosting_config["weak_learner_key"]
weak_learner_config = modeling_config[weak_learner_key]

processor = Processor(**active_dataset_config)

train, valid, test = data_manager.load_image_data(
    active_dataset
)

### ACCURACY ON CONVOLUTIONAL GRADIENT BOOSTING

In [5]:
preds_dir = f"{MODELS_PATH}/{active_dataset}/2026_03_24_12_25/predictions.csv"
preds = pd.read_csv(preds_dir).astype(int)[active_dataset_config["target"]].values

y_test = np.array(test.dataset.targets)[test.indices]

accuracy_score(y_test, preds)

0.7035

### LENET-5

In [24]:
test_numpy = processor.convert_to_numpy(test) 

raw_preds = vgg.predict(test_numpy) 

if len(raw_preds.shape) > 1 and raw_preds.shape[1] > 1:
    preds = np.argmax(raw_preds, axis=1)
else:
    preds = raw_preds.flatten().astype(int)


score = accuracy_score(y_test, preds)
print(f"Accuracy: {score:.4f}")

Accuracy: 0.7245


In [25]:
test_numpy = processor.convert_to_numpy(test) 

raw_preds = lenet5.predict(test_numpy) 

if len(raw_preds.shape) > 1 and raw_preds.shape[1] > 1:
    preds = np.argmax(raw_preds, axis=1)
else:
    preds = raw_preds.flatten().astype(int)


score = accuracy_score(y_test, preds)
print(f"Accuracy: {score:.4f}")

Accuracy: 0.9816


In [24]:
class LeNet5(CNNRegressor):
    def __init__(self):
        super().__init__(5, 4, 6, 4, batch_size=32)

    def fit(self, X_train, y_train, X_valid, y_valid, epochs=100, patience=10):
        
        num_classes = int(y_train.max() + 1)
        in_channels = X_train.shape[1]

        self._get_network(in_channels, num_classes)

        train_loader = self._prepare_loader(X_train, y_train)
        
        X_valid_t = torch.from_numpy(X_valid).to(torch.float32).to(self.device)
        y_valid_t = torch.from_numpy(y_valid).long().to(self.device).view(-1)
        
        criterion = nn.CrossEntropyLoss()
        optimizer = optim.Adam(self.parameters())
        
        best_loss = float('inf')
        best_model = None
        early_stop_count = 0

        for epoch in range(epochs):
            self.train()
            for batch_X, batch_y in train_loader:
                batch_X, batch_y = batch_X.to(self.device), batch_y.to(self.device)
                
                optimizer.zero_grad()
                preds = self(batch_X)
                
                loss = criterion(preds, batch_y.long().view(-1))
                loss.backward()
                optimizer.step()

            self.eval()
            with torch.no_grad():
                val_preds = self(X_valid_t)
                val_loss = criterion(val_preds, y_valid_t).item()

            if (epoch + 1) % 1 == 0:
                print(f"Epoch {epoch + 1}: Val Loss {val_loss:.4f}")

            if val_loss < best_loss:
                best_loss = val_loss
                best_model = copy.deepcopy(self.state_dict())
                early_stop_count = 0
            else:
                early_stop_count += 1

            if early_stop_count >= patience:
                break
                
        if best_model:
            self.load_state_dict(best_model)

    def _get_network(self, in_channels: int, output_size: int) -> None:
        self.network = nn.Sequential(
            # Layer 1: Conv 1
            nn.Conv2d(1, 3, kernel_size=5),  # input channels=1 (grayscale), output=6
            nn.Tanh(),                        # original LeNet uses tanh
            nn.AvgPool2d(kernel_size=2, stride=2),  # pooling layer
            
            # Layer 2: Conv 2
            nn.Conv2d(3, 3, kernel_size=5),
            nn.Tanh(),
            nn.AvgPool2d(kernel_size=2, stride=2),
            
            # Flatten for fully connected layers
            nn.Flatten(),
            
            # Fully connected layers
            nn.Linear(3*4*4, 40),
            nn.Tanh(),
            
            nn.Linear(40, 20),
            nn.Tanh(),
            
            nn.Linear(20, 10)  # 10 output classes
        )
        self.network.to(self.device)

X_train, y_train = processor.split_features_target(train)
X_valid, y_valid = processor.split_features_target(valid)

lenet5 = LeNet5()
lenet5.fit(X_train, y_train, X_valid, y_valid)

Epoch 1: Val Loss 0.2748
Epoch 2: Val Loss 0.1896
Epoch 3: Val Loss 0.1503
Epoch 4: Val Loss 0.1334
Epoch 5: Val Loss 0.1236
Epoch 6: Val Loss 0.1145
Epoch 7: Val Loss 0.1041
Epoch 8: Val Loss 0.0992
Epoch 9: Val Loss 0.0938
Epoch 10: Val Loss 0.0896
Epoch 11: Val Loss 0.0903
Epoch 12: Val Loss 0.0839
Epoch 13: Val Loss 0.0808
Epoch 14: Val Loss 0.0826
Epoch 15: Val Loss 0.0839
Epoch 16: Val Loss 0.0767
Epoch 17: Val Loss 0.0779
Epoch 18: Val Loss 0.0763
Epoch 19: Val Loss 0.0793
Epoch 20: Val Loss 0.0761
Epoch 21: Val Loss 0.0899
Epoch 22: Val Loss 0.0752
Epoch 23: Val Loss 0.0786
Epoch 24: Val Loss 0.0768
Epoch 25: Val Loss 0.0706
Epoch 26: Val Loss 0.0767
Epoch 27: Val Loss 0.0697
Epoch 28: Val Loss 0.0707
Epoch 29: Val Loss 0.0731
Epoch 30: Val Loss 0.0787
Epoch 31: Val Loss 0.0782
Epoch 32: Val Loss 0.0757
Epoch 33: Val Loss 0.0732
Epoch 34: Val Loss 0.0761
Epoch 35: Val Loss 0.0730
Epoch 36: Val Loss 0.0742
Epoch 37: Val Loss 0.0743


In [4]:
class SimpleCNN(CNNRegressor):
    def __init__(self):
        super().__init__(epochs=30, kernel_size=5, hidden_channels=4, max_pool_size=4, batch_size=32)

    def fit(self, X_train, y_train, X_valid, y_valid, epochs=100, patience=10):
        
        num_classes = int(y_train.max() + 1)
        in_channels = X_train.shape[1]

        self._get_network(in_channels, num_classes)

        train_loader = self._prepare_loader(X_train, y_train)
        
        X_valid_t = torch.from_numpy(X_valid).to(torch.float32).to(self.device)
        y_valid_t = torch.from_numpy(y_valid).long().to(self.device).view(-1)
        
        criterion = nn.CrossEntropyLoss()
        optimizer = optim.Adam(self.parameters())
        
        best_loss = float('inf')
        best_model = None
        early_stop_count = 0

        for epoch in range(self.epochs):
            self.train()
            for batch_X, batch_y in train_loader:
                batch_X, batch_y = batch_X.to(self.device), batch_y.to(self.device)
                
                optimizer.zero_grad()
                preds = self(batch_X)
                
                loss = criterion(preds, batch_y.long().view(-1))
                loss.backward()
                optimizer.step()

            self.eval()
            with torch.no_grad():
                val_preds = self(X_valid_t)
                val_loss = criterion(val_preds, y_valid_t).item()

            if (epoch + 1) % 1 == 0:
                print(f"Epoch {epoch + 1}: Val Loss {val_loss:.4f}")

            if val_loss < best_loss:
                best_loss = val_loss
                best_model = copy.deepcopy(self.state_dict())
                early_stop_count = 0
            else:
                early_stop_count += 1

            if early_stop_count >= patience:
                break
                
        if best_model:
            self.load_state_dict(best_model)

    def _get_network(self, in_channels: int, output_size: int) -> None:
        self.network = nn.Sequential(
            nn.Conv2d(3, 8, kernel_size=3),
            nn.ReLU(),
            nn.MaxPool2d(2),
            
            nn.Conv2d(8, 16, kernel_size=3),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(16, 16, kernel_size=3),
            nn.ReLU(),

            nn.Flatten(),
            
            nn.Linear(256, 16),
            nn.ReLU(),
           
            nn.Linear(16, 10)
        )
        self.network.to(self.device)

X_train, y_train = processor.split_features_target(train)
X_valid, y_valid = processor.split_features_target(valid)

simple_cnn = SimpleCNN()
simple_cnn.fit(X_train, y_train, X_valid, y_valid)

Epoch 1: Val Loss 1.6903
Epoch 2: Val Loss 1.5534
Epoch 3: Val Loss 1.4742
Epoch 4: Val Loss 1.4419
Epoch 5: Val Loss 1.4072
Epoch 6: Val Loss 1.4350
Epoch 7: Val Loss 1.3698
Epoch 8: Val Loss 1.3556
Epoch 9: Val Loss 1.3314
Epoch 10: Val Loss 1.3106
Epoch 11: Val Loss 1.3413
Epoch 12: Val Loss 1.2956
Epoch 13: Val Loss 1.2820
Epoch 14: Val Loss 1.2757
Epoch 15: Val Loss 1.2799
Epoch 16: Val Loss 1.2880
Epoch 17: Val Loss 1.2503
Epoch 18: Val Loss 1.2504
Epoch 19: Val Loss 1.2294
Epoch 20: Val Loss 1.2213
Epoch 21: Val Loss 1.2435
Epoch 22: Val Loss 1.2042
Epoch 23: Val Loss 1.2003
Epoch 24: Val Loss 1.2011
Epoch 25: Val Loss 1.1990
Epoch 26: Val Loss 1.2034
Epoch 27: Val Loss 1.1917
Epoch 28: Val Loss 1.1891
Epoch 29: Val Loss 1.1824
Epoch 30: Val Loss 1.1690


In [9]:
class VGGNet(CNNRegressor):
    def __init__(self):
        super().__init__(epochs=100, kernel_size=5, hidden_channels=4, max_pool_size=4, batch_size=32)

    def fit(self, X_train, y_train, X_valid, y_valid, epochs=100, patience=10):
        
        num_classes = int(y_train.max() + 1)
        in_channels = X_train.shape[1]

        self._get_network(in_channels, num_classes)

        train_loader = self._prepare_loader(X_train, y_train)
        
        X_valid_t = torch.from_numpy(X_valid).to(torch.float32).to(self.device)
        y_valid_t = torch.from_numpy(y_valid).long().to(self.device).view(-1)
        
        criterion = nn.CrossEntropyLoss()
        optimizer = optim.Adam(self.parameters())

        best_loss = float('inf')
        best_model = None
        early_stop_count = 0

        for epoch in range(epochs):
            self.train()
            for batch_X, batch_y in train_loader:
                batch_X, batch_y = batch_X.to(self.device), batch_y.to(self.device)
                
                optimizer.zero_grad()
                preds = self(batch_X)
                
                loss = criterion(preds, batch_y.long().view(-1))
                loss.backward()
                optimizer.step()

            self.eval()
            with torch.no_grad():
                val_preds = self(X_valid_t)
                val_loss = criterion(val_preds, y_valid_t).item()

            if (epoch + 1) % 1 == 0:
                print(f"Epoch {epoch + 1}: Val Loss {val_loss:.4f}")

            if val_loss < best_loss:
                best_loss = val_loss
                best_model = copy.deepcopy(self.state_dict())
                early_stop_count = 0
            else:
                early_stop_count += 1

            if early_stop_count >= patience:
                break
                
        if best_model:
            self.load_state_dict(best_model)

    def _get_network(self, in_channels: int, output_size: int) -> None:
        
        self.network = nn.Sequential(
            nn.Conv2d(3, 32, 3, padding=1),
            nn.ReLU(),
            nn.BatchNorm2d(32),

            nn.Conv2d(32, 32, 3, padding=1),
            nn.ReLU(),
            nn.BatchNorm2d(32),

            nn.MaxPool2d(2),
            nn.Dropout(0.2),

            nn.Conv2d(32, 64, 3, padding=1),
            nn.ReLU(),
            nn.BatchNorm2d(64),

            nn.Conv2d(64, 64, 3, padding=1),
            nn.ReLU(),
            nn.BatchNorm2d(64),

            nn.MaxPool2d(2),
            nn.Dropout(0.3),

            nn.Conv2d(64, 128, 3, padding=1),
            nn.ReLU(),
            nn.BatchNorm2d(128),

            nn.Conv2d(128, 128, 3, padding=1),
            nn.ReLU(),
            nn.BatchNorm2d(128),

            nn.MaxPool2d(2),
            nn.Dropout(0.4),

            nn.Flatten(),

            nn.Linear(128*4*4, 128),
            nn.ReLU(),
            nn.BatchNorm1d(128),
            nn.Dropout(0.5),

            nn.Linear(128, 10)
        )

        self.network.to(self.device)

X_train, y_train = processor.split_features_target(train)
X_valid, y_valid = processor.split_features_target(valid)

vgg = VGGNet()
vgg.fit(X_train, y_train, X_valid, y_valid)

Epoch 1: Val Loss 1.0949
Epoch 2: Val Loss 0.8019
Epoch 3: Val Loss 0.6977
Epoch 4: Val Loss 0.6404
Epoch 5: Val Loss 0.6100
Epoch 6: Val Loss 0.5896
Epoch 7: Val Loss 0.5415
Epoch 8: Val Loss 0.6205
Epoch 9: Val Loss 0.5436
Epoch 10: Val Loss 0.4937
Epoch 11: Val Loss 0.4733
Epoch 12: Val Loss 0.4674
Epoch 13: Val Loss 0.4805
Epoch 14: Val Loss 0.4579
Epoch 15: Val Loss 0.4646
Epoch 16: Val Loss 0.5041
Epoch 17: Val Loss 0.4387
Epoch 18: Val Loss 0.4448
Epoch 19: Val Loss 0.4451
Epoch 20: Val Loss 0.4476
Epoch 21: Val Loss 0.4322
Epoch 22: Val Loss 0.4366
Epoch 23: Val Loss 0.4369


KeyboardInterrupt: 